In [12]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
from IPython.core.pylabtools import figsize
from scipy.signal import butter, filtfilt

In [13]:
path = '/Users/paul/Coding_Projects/playagain/data/playagain/muovi/data_per_subject/VP_01/cleaned/session_5_cleaned.csv'

In [14]:
df = pd.read_csv(path)

In [15]:
df.keys()

Index(['Timestamp', 'RMS', 'GestureActive', 'GroundTruth', 'CameraBlocking',
       'InputSource', 'EMG_Ch0', 'EMG_Ch1', 'EMG_Ch2', 'EMG_Ch3', 'EMG_Ch4',
       'EMG_Ch5', 'EMG_Ch6', 'EMG_Ch7', 'EMG_Ch8', 'EMG_Ch9', 'EMG_Ch10',
       'EMG_Ch11', 'EMG_Ch12', 'EMG_Ch13', 'EMG_Ch14', 'EMG_Ch15', 'EMG_Ch16',
       'EMG_Ch17', 'EMG_Ch18', 'EMG_Ch19', 'EMG_Ch20', 'EMG_Ch21', 'EMG_Ch22',
       'EMG_Ch23', 'EMG_Ch24', 'EMG_Ch25', 'EMG_Ch26', 'EMG_Ch27', 'EMG_Ch28',
       'EMG_Ch29', 'EMG_Ch30', 'EMG_Ch31', 'Time'],
      dtype='object')

In [16]:
df.head()

,Timestamp,RMS,GestureActive,GroundTruth,CameraBlocking,InputSource,EMG_Ch0,EMG_Ch1,EMG_Ch2,EMG_Ch3,...,EMG_Ch23,EMG_Ch24,EMG_Ch25,EMG_Ch26,EMG_Ch27,EMG_Ch28,EMG_Ch29,EMG_Ch30,EMG_Ch31,Time
0,0.001015,1.539885e-07,0.0,-8.611047e-123,2.380678e-300,EMG,-4.574421e-07,-0.000002,1.085159e-07,-7.511938e-07,...,-0.000002,-0.000002,-9.867976e-07,8.242408e-08,-6.556797e-07,-8.127549e-08,6.547978e-07,-1.000361e-07,-0.000002,0.0000
1,0.001515,1.539885e-07,0.0,-8.611047e-123,2.380678e-300,EMG,-4.574421e-07,-0.000002,1.085159e-07,-7.511938e-07,...,-0.000002,-0.000002,-9.867976e-07,8.242408e-08,-6.556797e-07,-8.127549e-08,6.547978e-07,-1.000361e-07,-0.000002,0.0005
2,0.002015,1.539885e-07,0.0,-8.611047e-123,2.380678e-300,EMG,-4.574421e-07,-0.000002,1.085159e-07,-7.511938e-07,...,-0.000002,-0.000002,-9.867976e-07,8.242408e-08,-6.556797e-07,-8.127549e-08,6.547978e-07,-1.000361e-07,-0.000002,0.0010
3,0.002515,1.539885e-07,0.0,-8.611047e-123,2.380678e-300,EMG,-4.574421e-07,-0.000002,1.085159e-07,-7.511938e-07,...,-0.000002,-0.000002,-9.867976e-07,8.242408e-08,-6.556797e-07,-8.127549e-08,6.547978e-07,-1.000361e-07,-0.000002,0.0015
4,0.003015,1.539885e-07,0.0,-8.611047e-123,2.380678e-300,EMG,-4.574421e-07,-0.000002,1.085159e-07,-7.511938e-07,...,-0.000002,-0.000002,-9.867976e-07,8.242408e-08,-6.556797e-07,-8.127549e-08,6.547978e-07,-1.000361e-07,-0.000002,0.0020


In [17]:
# compute RMS of summed channels
# - select numeric columns, try to exclude obvious time/index columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
time_like = [c for c in numeric_cols if any(t in c.lower() for t in ('time', 'timestamp', 'sample', 'index'))]
channels = [c for c in numeric_cols if c not in time_like]
if len(channels) == 0:
    # fallback to using all numeric columns if no channel-like column names found
    channels = numeric_cols

# create summed signal across channels (row-wise sum)
df['sum_channels'] = df[channels].sum(axis=1)

# RMS of the summed signal (scalar)
rms_sum = np.sqrt(np.mean(df['sum_channels'] ** 2))